### Propagación de capacidad y velocidad a lo largo de las avenidas

Este notebook toma los enlaces de OSMNX que ya recibieron atributos de TransCAD en los notebooks anteriores y extiende esa información a lo largo de las avenidas principales.

La idea es aprovechar los tramos que sí hicieron match para completar los segmentos que quedaron sin capacidad o velocidad, pero que pertenecen a la misma avenida.

In [84]:
import pandas as pd
import geopandas as gpd
import numpy as np
import os

In [89]:
folder = r'/Users/jeannettearjona/Library/CloudStorage/OneDrive-InstitutoTecnologicoydeEstudiosSuperioresdeMonterrey/Modelación Urbana - Red Vial Guadalajara/VisumLinks With TransCAD Atts'
output_02_cleaned = "OSM Links con TransCAD Atts 01 Cleaned"
# osm links y algunas partes de av. princiaples tienen capacidad y vel
osm_links = gpd.read_file(os.path.join(folder, output_02_cleaned, "edges_osmnx_con_vel_y_cap_cleaned.shp"))

In [90]:
#Cols
nombre = "name"
capacidad = "CAPACIDAD"
vel_prom = "Velocidad_"
v0 = "Limite_vel"
carriles = "CARRILES"

In [88]:
cols = ["CAPACIDAD", "Limite_vel", "Velocidad_", "CARRILES"]

conteo = pd.DataFrame({
    "columna": cols,
    "con_dato": [osm_links[c].notna().sum() for c in cols],
    "sin_dato": [osm_links[c].isna().sum() for c in cols],
    "total": [len(osm_links)] * len(cols)
})

conteo

,columna,con_dato,sin_dato,total
0,CAPACIDAD,7158,487738,494896
1,Limite_vel,7158,487738,494896
2,Velocidad_,7158,487738,494896
3,CARRILES,11824,483072,494896


### Separación de enlaces con y sin atributos

Primero se dividen los links de OSMNX en dos grupos:

- los que ya traen capacidad y velocidad heredadas de TransCAD;
- los que todavía no tienen esos atributos.

Esto permite revisar qué avenidas sí tienen información suficiente para propagar atributos y cuáles todavía quedan incompletas.

In [92]:
#links con nombre
links_with_name = osm_links[osm_links[nombre].notnull()].copy()

#links con NOMBRE y CAPACIDAD y VELOCIDAD (heredadas de transcad)
links_con_atts = links_with_name[
    links_with_name[capacidad].notna() &
    links_with_name[vel_prom].notna() &
    links_with_name[v0].notna() 
].copy()

#links con NOMBRE pero sin capacidad y velocidad (heredadas de transcad)
links_sin_atts = links_with_name[
    links_with_name[capacidad].isna() &
    links_with_name[vel_prom].isna() &
    links_with_name[v0].isna() 
].copy()

#ALL links con capacidad y velocidad
all_links_con_atts = osm_links[
    osm_links[capacidad].notna() &
    osm_links[vel_prom].notna() &
    osm_links[v0].notna() 
].copy()

print(f"Total OSMNX Links: {len(osm_links)}")
print(f"Todos los links (con o sin nombre) con CAPACIDAD y VELOCIDAD heredadas de TransCAD: {len(all_links_con_atts)}")
print(f"Links con nombre, CAPACIDAD y VELOCIDAD: {len(links_con_atts)}")

Total OSMNX Links: 494896
Todos los links (con o sin nombre) con CAPACIDAD y VELOCIDAD heredadas de TransCAD: 7220
Links con nombre, CAPACIDAD y VELOCIDAD: 6852


### Revisión del tipo de vialidad

Antes de propagar atributos, se revisa qué tipo de `highway` tienen los enlaces que ya recibieron información de TransCAD.  
Esto ayuda a confirmar si los atributos están cayendo sobre avenidas principales y no sobre vías secundarias o no deseadas.

In [93]:
# ¿Que tipo de vialidad tienen los links que heredaron atributo de TransCAD en notebooks 01 y 02?
all_links_con_atts['highway'].value_counts(dropna=False)

highway
primary                 2875
secondary               2097
tertiary                1216
trunk                    569
unclassified             265
motorway                 187
track                     10
['primary', 'trunk']       1
Name: count, dtype: int64

In [94]:
# ¿Que tipo de vialidad tienen los links que heredaron atributo de TransCAD en notebooks 01 y 02 Y TIENEN NOMBRE?
links_con_atts['highway'].value_counts(dropna=False)

highway
primary                 2827
secondary               2019
tertiary                1122
trunk                    564
motorway                 186
unclassified             125
track                      8
['primary', 'trunk']       1
Name: count, dtype: int64

### Búsqueda de avenidas que pueden completarse por nombre

Los enlaces sin atributos se comparan contra los que sí tienen atributos para ver si comparten el mismo nombre de avenida.

Si varios fragmentos pertenecen a la misma vía, entonces se pueden completar los que faltan usando el comportamiento típico de esa avenida.

In [95]:
# ¿cuantos nombres de los links_sin_atts existen en links_con_atts?
nombres_con_atts = set(links_con_atts[nombre])
nombres_sin_atts = set(links_sin_atts[nombre])

# links sin atts que cuyo nombre existe en links con atts
links_rellenables_por_nombre = links_sin_atts[
    links_sin_atts[nombre].isin(nombres_con_atts)
].copy()

nombres_match = nombres_sin_atts.intersection(nombres_con_atts)

print(f"{len(nombres_match)} nombres únicos pueden servir para propagación")
print(f"Esos {len(nombres_match)} cubren {len(links_rellenables_por_nombre)} links sin attributos")


242 nombres únicos pueden servir para propagación
Esos 242 cubren 24366 links sin attributos


### Verificación de consistencia por avenida

Se revisa si todos los fragmentos que comparten el mismo nombre tienen los mismos valores de capacidad y velocidad.

Esto es importante porque algunas avenidas están divididas en varios enlaces y no siempre todos representan la misma jerarquía vial o el mismo comportamiento de circulación.

### ¿Todos los links que tienen el mismo nombre tienen __una sola__ capacidad y velocidad a lo largo de la via?

In [96]:
# cuantos diferentes valores de capacidad y velocidad existen en cada via (groupby=name)
revision_nombres = (
    links_con_atts
    .groupby(nombre)
    .agg(
        n_cap=(capacidad, "nunique"),
        n_vel=(vel_prom, "nunique"),
        n_v0=(v0, "nunique")
    )
    .reset_index()
)

# cuales son aquellos nombres de vias que tienen mas de un valor de capacidad o velocidad para la misma via
nombres_inconsistentes = revision_nombres[
    (revision_nombres["n_cap"] > 1) |
    (revision_nombres["n_vel"] > 1) |
    (revision_nombres["n_v0"] > 1)
]

print(f"{len(nombres_inconsistentes)} links diferentes valores de capacidad y velocidad para el mismo nombre")

215 links diferentes valores de capacidad y velocidad para el mismo nombre


### Caso de ejemplo

Se revisa una avenida específica para ver cómo varían sus valores de capacidad, velocidad promedio y límite de velocidad entre sus fragmentos.

Este ejemplo sirve para justificar por qué se usa una mediana por nombre en lugar de conservar valores distintos en cada fragmento.

Las variaciones en capacidad no son confiables porque vienen de una red con geometrías excesivamente agregadas entonces esos cambios de capacidad es porque un solo link pasa de representar a 2 vias a representar a 4 o 6

In [97]:
#Case study
nombre_prueba = "Avenida Doctor Roberto Michel"

links_con_atts.loc[
    links_con_atts[nombre] == nombre_prueba,
    [nombre, capacidad, vel_prom, v0]
].drop_duplicates()

,name,CAPACIDAD,Velocidad_,Limite_vel
101665,Avenida Doctor Roberto Michel,12000.0,26.000000,40.0
101666,Avenida Doctor Roberto Michel,12000.0,26.000007,40.0
101675,Avenida Doctor Roberto Michel,12000.0,25.999950,40.0
101679,Avenida Doctor Roberto Michel,12000.0,26.000057,40.0
101688,Avenida Doctor Roberto Michel,12000.0,26.000079,40.0
101701,Avenida Doctor Roberto Michel,8000.0,26.000047,40.0
102023,Avenida Doctor Roberto Michel,9000.0,21.999453,40.0
105487,Avenida Doctor Roberto Michel,9000.0,22.000052,40.0
105530,Avenida Doctor Roberto Michel,6000.0,21.999988,40.0
105534,Avenida Doctor Roberto Michel,6000.0,22.000181,40.0


### Expansión longitudinal por nombre

Para cada avenida con nombre se calcula un valor típico de capacidad, velocidad promedio y límite de velocidad.

La mediana se usa como valor representativo porque reduce el efecto de fragmentos atípicos y permite asignar un solo valor a toda la avenida.

### __usar la mediana por nombre__ para capacidad, velocidad, v0

In [98]:
# "Para cada avenida voy a usar el valor típico que ya existe en esa avenida."

#1) Agrupar por nombre usando la MEDIANA de: capacidad, vel_prom, v0
atts_por_nombre = (
    links_con_atts
    .groupby(nombre)[[capacidad, vel_prom, v0]]
    .median()
    .reset_index()
    .rename(columns={
        capacidad: "cap_med", #med for "mediana"
        vel_prom: "velprom_med",
        v0: "limvel_med"
    })
)

#2) Merge a todos los links de osm
# "todos los que tengan el mismo nombre recibirán la mediana de capacidad y velocidad de los fragmentos de la via que pudieron heredar de transcad"
links_fill = osm_links.merge(
    atts_por_nombre, 
    on=nombre, 
    how='left'
)

In [99]:
# Ej. la misma avenida que tenia varios valores de capacidad y velocidad ahora tiene la misma mediana, de esta forma toda la avenida usa el mismo valor
atts_por_nombre[atts_por_nombre[nombre]=='Avenida Doctor Roberto Michel']

,name,cap_med,velprom_med,limvel_med
39,Avenida Doctor Roberto Michel,9000.0,25.99995,40.0


### Filtrado de atributos después del `merge` por nombre

Los atributos de capacidad y velocidad se propagaron a la red OSMNX mediante un `merge` usando el campo `nombre`.  
Sin embargo, en algunos casos una vialidad principal y una calle local comparten el mismo nombre, por ejemplo una avenida Benito Juarez y un tramo residencial que tambien se llama Benito Juarez.

Para evitar que una calle pequeña herede atributos de una avenida solo por tener el mismo nombre, después del `merge` se colocaron valores `NaN` en las columnas de capacidad y velocidad para todos los links cuyo tipo `highway` no corresponde a una vialidad principal.

De esta forma, solo conservan atributos los segmentos que realmente forman parte de la red vial principal.

In [100]:
# 3) Dejar NaN en links que no deben recibir atributos
highways_validos_para_recibir = [
    "primary",
    "secondary",
    "tertiary",
    "trunk",
    "unclassified",
    "motorway",
]

def highway_is_valid(hwy):
    if isinstance(hwy, list):
        return any(x in highways_validos_para_recibir for x in hwy)
    return hwy in highways_validos_para_recibir

mask_validos = links_fill["highway"].apply(highway_is_valid)

links_fill.loc[
    ~mask_validos,
    ["cap_med", "velprom_med", "limvel_med"]
] = np.nan

In [101]:
links_fill[links_fill['cap_med'].notna()]['highway'].value_counts()

highway
primary         7062
secondary       6711
tertiary        6329
trunk           1405
unclassified     952
motorway         367
Name: count, dtype: int64

### Construcción de atributos finales

Para los enlaces que sí recibieron atributos por expansión del nombre, se conservó la mediana calculada (`*_med`).  
Para los enlaces sin nombre o sin coincidencia por nombre, se conservaron los valores originales disponibles en la red OSMNX.

De esta forma, las columnas finales combinan la información propagada por nombre con los atributos originales cuando la expansión no fue posible.

In [55]:
links_fill["cap_final"] = links_fill["cap_med"].fillna(links_fill[capacidad])
links_fill["velprom_final"] = links_fill["velprom_med"].fillna(links_fill[vel_prom])
links_fill["limvel_final"] = links_fill["limvel_med"].fillna(links_fill[v0])

In [56]:
# Aquellos links que si heredadon capacidad y velocidad de TransCAD desde un inicio pero no participaron en la expansion al no tener nombre
# Conservar su capacidad y velocidad originalmente heredadas
links_fill[(links_fill['CAPACIDAD'].notna()) & (links_fill['cap_med'].isna())].head()

,u,v,key,osmid,highway,lanes,name,oneway,ref,reversed,...,CAPACIDAD,Velocidad_,Limite_vel,geometry,cap_med,velprom_med,limvel_med,cap_final,velprom_final,limvel_final
514,1458419082,1458419090,0,132600592,secondary,2,NaN,True,NaN,False,...,3000.0,22.000020,40.0,"LINESTRING (-103.2511 20.62594, -103.25102 20....",NaN,NaN,NaN,3000.0,22.000020,40.0
523,1458419090,1361070508,0,132600592,secondary,2,NaN,True,NaN,False,...,3000.0,21.999944,40.0,"LINESTRING (-103.25071 20.62605, -103.2507 20....",NaN,NaN,NaN,3000.0,21.999944,40.0
2947,1639966364,5788237647,0,611329210,tertiary,1,NaN,True,NaN,False,...,2000.0,25.999522,40.0,"LINESTRING (-103.24091 20.61625, -103.24106 20...",NaN,NaN,NaN,2000.0,25.999522,40.0
8297,1746083425,4998594004,0,510866828,primary,NaN,NaN,True,MEX 90,False,...,2500.0,53.999559,80.0,"LINESTRING (-103.05099 20.62219, -103.05273 20...",NaN,NaN,NaN,2500.0,53.999559,80.0
8313,1746083518,4998593972,0,"[510866833, 188992265, 732805806]",trunk,2,NaN,True,MEX 80;MEX 70;MEX 90,False,...,5000.0,54.000124,80.0,"LINESTRING (-103.05308 20.62265, -103.0532 20....",NaN,NaN,NaN,5000.0,54.000124,80.0


In [57]:
links_fill.head(10)

,u,v,key,osmid,highway,lanes,name,oneway,ref,reversed,...,CAPACIDAD,Velocidad_,Limite_vel,geometry,cap_med,velprom_med,limvel_med,cap_final,velprom_final,limvel_final
0,267537966,7306651630,0,"[832652768, 619339782, 688424559, 188974544, 1...",motorway,"['3', '2']",Autopista Guadalajara - Morelia,True,MEX 15D;MEX 80D,False,...,2000.0,34.000000,80.0,"LINESTRING (-103.24632 20.61606, -103.24734 20...",3250.0,34.000061,80.0,3250.0,34.000061,80.0
1,267537966,5837556433,0,694566994,motorway_link,1,NaN,True,NaN,False,...,NaN,NaN,NaN,"LINESTRING (-103.24632 20.61606, -103.24648 20...",NaN,NaN,NaN,NaN,NaN,NaN
2,267538751,273140976,0,189118222,motorway,2,Autopista Guadalajara - Zapotlanejo,True,MEX 80D;MEX 90D,False,...,NaN,NaN,NaN,"LINESTRING (-103.1364 20.60565, -103.13432 20....",5000.0,53.999989,80.0,5000.0,53.999989,80.0
3,267538751,1746293763,0,907206852,motorway_link,1,Autopista Guadalajara - Morelia,True,MEX 15D,False,...,NaN,NaN,NaN,"LINESTRING (-103.1364 20.60565, -103.1354 20.6...",NaN,NaN,NaN,NaN,NaN,NaN
4,273140976,1997658287,0,835083267,motorway,3,Autopista Guadalajara - Zapotlanejo,True,MEX 80D;MEX 90D,False,...,NaN,NaN,NaN,"LINESTRING (-103.13185 20.60758, -103.13162 20...",5000.0,53.999989,80.0,5000.0,53.999989,80.0
5,296347106,4670183699,0,189099938,trunk,2,Carretera Guadalajara - Entronque Jocotepec,True,MEX 23,False,...,NaN,NaN,NaN,"LINESTRING (-103.2612 20.47119, -103.26095 20....",4000.0,47.000261,80.0,4000.0,47.000261,80.0
6,296347106,5000459113,0,1020599956,service,NaN,NaN,False,NaN,True,...,NaN,NaN,NaN,"LINESTRING (-103.2612 20.47119, -103.26134 20....",NaN,NaN,NaN,NaN,NaN,NaN
7,296347259,6147898453,0,189099937,trunk,2,Carretera Guadalajara - Entronque Jocotepec,True,MEX 23,False,...,5000.0,54.000045,80.0,"LINESTRING (-103.24953 20.45228, -103.24815 20...",4000.0,47.000261,80.0,4000.0,47.000261,80.0
8,296347259,1710496522,0,189114169,trunk_link,NaN,NaN,True,NaN,False,...,NaN,NaN,NaN,"LINESTRING (-103.24953 20.45228, -103.2494 20....",NaN,NaN,NaN,NaN,NaN,NaN
9,296347282,3964030763,0,27312392,trunk_link,NaN,NaN,True,NaN,False,...,NaN,NaN,NaN,"LINESTRING (-103.23283 20.41315, -103.23285 20...",NaN,NaN,NaN,NaN,NaN,NaN


In [59]:
#ALL links con capacidad y velocidad
final_links_con_atts = links_fill[
    links_fill['cap_final'].notna() &
    links_fill['velprom_final'].notna() &
    links_fill['limvel_final'].notna()
].copy()

print(f"Antes de la expansión {len(all_links_con_atts)} tenian velocidad y capacidad heredados")
print(f"Tras la expansión {len(final_links_con_atts)} tienen velocidad y capacidad heredados/expandidos")

Antes de la expansión 7220 tenian velocidad y capacidad heredados
Tras la expansión 23203 tienen velocidad y capacidad heredados/expandidos


In [60]:
output_folder = "OSM Links con velocidad y capacidades (a lo largo)"
links_fill.to_file(os.path.join(folder, output_folder, "osm_links_con_atts_a_lo_largo.shp"))

/var/folders/8s/00_wwq9j23b09mnd7gp105m00000gp/T/ipykernel_68696/2417806747.py:2: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  links_fill.to_file(os.path.join(folder, output_folder, "osm_links_con_atts_a_lo_largo.shp"))
/opt/anaconda3/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'velprom_med' to 'velprom_me'
  ogr_write(
/opt/anaconda3/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'velprom_final' to 'velprom_fi'
  ogr_write(
/opt/anaconda3/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'limvel_final' to 'limvel_fin'
  ogr_write(
